# 4. Passage Analysis

Explore passage-level abstraction scores: find extreme passages, compare genres,
and generate marked-up text showing which words are abstract/concrete.

In [ ]:
import pandas as pd
from IPython.display import HTML, Markdown, display
from abstraction.scoring import (
    get_all_passages, sample_passages, score_psg,
    gen_bookpassages, printpsg,
)
from abstraction.corpus import load_corpus

## Load all passage scores

In [ ]:
df = get_all_passages('CanonFiction')
print(f'{len(df)} passages')
df[['id', 'slice', 'num_abs', 'num_conc', 'abs-conc', 'zbin', 'year']].head()

## Most abstract passages

In [ ]:
meta = load_corpus('CanonFiction').metadata
dfm = df.merge(meta[['id', 'author', 'title', 'year', 'major_genre']], on='id', suffixes=('', '_meta'))

most_abstract = dfm.sort_values('abs-conc', ascending=False).head(10)
most_abstract[['author', 'title', 'year_meta', 'major_genre', 'num_abs', 'num_conc', 'abs-conc']]

## Most concrete passages

In [ ]:
most_concrete = dfm.sort_values('abs-conc').head(10)
most_concrete[['author', 'title', 'year_meta', 'major_genre', 'num_abs', 'num_conc', 'abs-conc']]

## Stratified sampling

Sample passages across time periods and abstraction levels for balanced analysis.

In [ ]:
sample = sample_passages(df, n=5)
print(f'{len(sample)} sampled passages')
sample.groupby(['ybin', 'zbin']).size().unstack(fill_value=0)

## Generate passages for a single text

Score every window in a single novel, with marked-up passage text.

In [ ]:
# Pick a text ID from the corpus
corpus = load_corpus('CanonFiction')
corpus.metadata[['id', 'author', 'title', 'year']].sample(5)

In [ ]:
# Uncomment and set a text_id to generate passages:
# text_id = 'Richardson.Pamela'  # example
# book_df = gen_bookpassages('CanonFiction', text_id)
# book_df.sort_values('abs-conc', ascending=False).head()

## Quick ad-hoc scoring

Paste any text and score it.

In [ ]:
from abstraction.counting import count_absconc_psg

my_text = """
The question of whether machines can think is itself a question about the
nature of thought, consciousness, and understanding. These abstract concepts
resist easy definition, and any answer we give reveals more about our own
assumptions than about the machines themselves.
"""

result = count_absconc_psg(my_text)
if len(result):
    row = result.iloc[-1]  # most abstract window
    display(HTML(f'<p>{row["passage"]}</p>'))
    print(f'Abs: {row["num_abs"]}, Conc: {row["num_conc"]}, Abs-Conc: {row["abs-conc"]}')
else:
    print(f'Score: {score_psg(my_text):.3f} (text too short for windowed counting)')